# Session 05 · MMS automatic speech recognition in Colab

**Student workbook — Data Creation II: ASR**

This workbook teaches you to find a speech dataset, **run an existing MMS model**, prepare audio/transcription pairs, **fine-tune only the MMS adapters and output head**, and examine character error rate (CER). It follows our class pipeline: **recording → segmentation → transcription → normalisation → model → recognition → evaluation**.

Choose **Runtime → Change runtime type → T4 GPU**, if one is available. MMS-1B needs several GB of downloads; a free Colab GPU is **not guaranteed**. The 12-step adapter run is educational and is *not* enough to create a useful general ASR system. Do not upload private recordings without permission.

Before loading data, add a Hugging Face token and uncomment **one** language and dataset choice. State the language you choose in your MMS work.

Sources: [MMS model card](https://huggingface.co/facebook/mms-1b-all) · [MMS adapter tutorial](https://huggingface.co/blog/mms_adapters) · [FLEURS dataset](https://huggingface.co/datasets/google/fleurs) · [Mozilla Common Voice](https://mozilladatacollective.com/datasets?q=common+voice) · [Colab FAQ](https://research.google.com/colaboratory/faq.html).

## 0 · Set up your environment

Colab already has Python and often PyTorch. We install compatible versions of Hugging Face Transformers, datasets and audio packages. This uses **Transformers 4.x** APIs. If Colab requests a restart, use **Runtime → Restart session**.

In [ ]:
%pip -q install "transformers==4.52.4" "datasets==3.6.0" "accelerate==1.7.0" "librosa==0.11.0" "soundfile==0.13.1" "jiwer==3.1.0" "safetensors>=0.5,<1"

In [ ]:
import torch, transformers, datasets, sys
print('Python:',sys.version.split()[0], 'Torch:',torch.__version__)
print('Transformers:',transformers.__version__, 'Datasets:',datasets.__version__)
print('GPU:',torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'No GPU available')

### Add your Hugging Face token

To run this workbook, add a Hugging Face token. [Create an account](https://huggingface.co/join), then open [Access Tokens](https://huggingface.co/settings/tokens), choose **New token**, name it, and give it the **read** role. Copy the token once. In Colab, click the **key icon (Secrets)** in the left sidebar, add a secret named `HF_TOKEN`, paste the token as its value, and enable **Notebook access** for this notebook. Run the check below before choosing a language. Never paste the token into a code cell or share it in a notebook copy.

FLEURS and the MMS checkpoint used here are public. The token authenticates your workbook to Hugging Face; the public FLEURS dataset itself does not require one. [Hugging Face token guidance](https://huggingface.co/docs/hub/security-tokens) · [Hugging Face Colab authentication guidance](https://huggingface.co/docs/huggingface_hub/en/quick-start#authentication).

In [ ]:
from google.colab import userdata
from huggingface_hub import login, whoami

try:
    hf_token = userdata.get('HF_TOKEN')
except Exception as exc:
    raise RuntimeError('Add HF_TOKEN in Colab Secrets and enable Notebook access, then rerun this cell.') from exc
if not hf_token:
    raise RuntimeError('HF_TOKEN is empty. Add a Hugging Face read token in Colab Secrets.')
try:
    account_name = whoami(token=hf_token)['name']
except Exception as exc:
    raise RuntimeError('Hugging Face rejected HF_TOKEN. Check the token in Colab Secrets and try again.') from exc
login(token=hf_token, add_to_git_credential=False, skip_if_logged_in=False)
print('Hugging Face authentication ready for:', account_name)
del hf_token

## 1 · Choose a language and dataset

The practical uses [FLEURS](https://huggingface.co/datasets/google/fleurs), a public speech dataset with audio, transcriptions, and train/validation/test partitions. In the next code cell, **uncomment exactly one** English, German, or Maltese line. Each line pairs a FLEURS dataset configuration with its corresponding **MMS language adapter code**. Do not mix these codes: FLEURS uses `en_us`, `de_de`, or `mt_mt`, while MMS uses `eng`, `deu`, or `mlt`. Check the [MMS model card](https://huggingface.co/facebook/mms-1b-all) for supported adapters.

Write the language you selected, the dataset, and the MMS adapter code in your notes and final reflection. Read the [FLEURS dataset card](https://huggingface.co/datasets/google/fleurs) for its licence, audio format, transcripts and intended use.

For a larger spoken corpus used in ASR work, explore [Mozilla Common Voice on Mozilla Data Collective](https://mozilladatacollective.com/datasets?q=common+voice). It has English, German and Maltese releases. Common Voice's old Hugging Face dataset repository no longer provides downloads; obtain it from Mozilla and check the release terms. This notebook uses small FLEURS samples to make the practical manageable in Colab.

If you use your own **permission-cleared** aligned recordings and transcripts, set `DATA_SOURCE='csv'`, `CUSTOM_LANGUAGE_NAME`, `CUSTOM_MMS_LANG`, and `CSV_PATH` below. A language need not have a pretrained MMS adapter to *try* a newly trained one, but 24 samples will not give a useful model for an unseen language.

In [ ]:
# === EDIT THESE SETTINGS ===
DATA_SOURCE = 'fleurs'     # change to 'csv' only for your own audio and transcripts

# Uncomment EXACTLY ONE language and dataset pair for the FLEURS practical.
FLEURS_CHOICES = []
# FLEURS_CHOICES.append(('English', 'en_us', 'eng'))
# FLEURS_CHOICES.append(('German', 'de_de', 'deu'))
# FLEURS_CHOICES.append(('Maltese', 'mt_mt', 'mlt'))

# For DATA_SOURCE='csv', fill both settings below instead of a FLEURS choice.
CUSTOM_LANGUAGE_NAME = ''
CUSTOM_MMS_LANG = ''      # supported MMS adapter code, if available
CSV_PATH = '/content/my_dataset/manifest.csv'

if DATA_SOURCE == 'fleurs':
    if len(FLEURS_CHOICES) != 1:
        raise ValueError('Uncomment exactly one English, German or Maltese FLEURS_CHOICES line above.')
    LANGUAGE_NAME, FLEURS_CONFIG, MMS_LANG = FLEURS_CHOICES[0]
elif DATA_SOURCE == 'csv':
    if FLEURS_CHOICES:
        raise ValueError('Leave FLEURS_CHOICES commented out when DATA_SOURCE is csv.')
    if not CUSTOM_LANGUAGE_NAME.strip() or not CUSTOM_MMS_LANG.strip():
        raise ValueError('Set CUSTOM_LANGUAGE_NAME and CUSTOM_MMS_LANG for your CSV data.')
    LANGUAGE_NAME, MMS_LANG = CUSTOM_LANGUAGE_NAME.strip(), CUSTOM_MMS_LANG.strip()
    FLEURS_CONFIG = None
else:
    raise ValueError("DATA_SOURCE must be 'fleurs' or 'csv'.")

print('Chosen language:', LANGUAGE_NAME, '| dataset:',
      f'FLEURS {FLEURS_CONFIG}' if DATA_SOURCE == 'fleurs' else CSV_PATH,
      '| MMS adapter:', MMS_LANG)
MMS_MODEL = 'facebook/mms-1b-all'
TRAIN_N, VALID_N, TEST_N = 24, 6, 6
MAX_SECONDS = 10.0
NORMALISE_NFC = True      # no deleting tone, aspiration, case, punctuation
RUN_BASELINE = True       # downloads the MMS 1B model
RUN_FINETUNING = False    # set True only when GPU is available
STEPS = 12               # educational miniature fine-tuning
LEARNING_RATE = 1e-3

## 2 · Load audio and reference transcriptions

For FLEURS we retain the dataset's official **train**, **validation** and **test** partitions. In research work, check whether the same speaker/recording appears in more than one partition. Keep the evaluation examples **unseen** during model training.

In [ ]:
import numpy as np
from datasets import load_dataset, Audio

def fleurs_rows(split, n):
    data = load_dataset('google/fleurs', FLEURS_CONFIG, split=f'{split}[:{n}]')
    data = data.cast_column('audio', Audio(sampling_rate=16000))
    return [{'audio':np.asarray(x['audio']['array'], dtype=np.float32), 'sr':16000,
             'text_original':x['transcription'], 'text_target':x['transcription'],
             'source_id':f"fleurs_{split}_{x['id']}"} for x in data]

if DATA_SOURCE == 'fleurs':
    train_rows = fleurs_rows('train', TRAIN_N)
    valid_rows = fleurs_rows('validation', VALID_N)
    test_rows = fleurs_rows('test', TEST_N)
    print('Rows:',len(train_rows),len(valid_rows),len(test_rows))

### Alternative: supply your own audio and transcript pairs

For `DATA_SOURCE='csv'` put a CSV file and audio (WAV/FLAC) in Google Drive or `/content/my_dataset/`. CSV fields are `audio_path,text_target,split`, with a `train`, `validation` or `test` value. Optional `text_original` preserves the source transcription.

```csv
audio_path,text_target,split
audio/001.wav,ma555,train
audio/002.wav,ma5,validation
audio/003.wav,ma555,test
```

This is **notation only**, not a claim that `ma5` and `ma555` mean the same tone. Choose normalisation only after analysing the language's transcription conventions. For ELAN `.eaf` source files see the repository's [EAF-to-CSV workflow](https://github.com/juanpalm-pixel/asr-work/blob/main/docs/eaf-pipeline.md). Do not use translations as ASR targets.

In [ ]:
from pathlib import Path
import csv, soundfile as sf, librosa

def read_custom(csv_path):
    csv_path = Path(csv_path)
    with csv_path.open(newline='', encoding='utf8') as stream:
        reader = csv.DictReader(stream)
        if not {'audio_path','text_target','split'}.issubset(set(reader.fieldnames or [])):
            raise ValueError('Required columns: audio_path,text_target,split')
        entries = list(reader)
    result = {'train':[], 'validation':[], 'test':[]}
    for row in entries:
        split=row['split'].strip().lower()
        if split not in result: raise ValueError('Unknown split: '+split)
        source=Path(row['audio_path'])
        if not source.is_absolute(): source=csv_path.parent/source
        audio,sr=sf.read(source)
        if audio.ndim>1: audio=audio.mean(axis=1)
        if sr!=16000: audio=librosa.resample(audio.astype(np.float32),orig_sr=sr,target_sr=16000)
        result[split].append({'audio':np.asarray(audio,dtype=np.float32),'sr':16000,
          'text_original':row.get('text_original') or row['text_target'],
          'text_target':row['text_target'],'source_id':str(source)})
    if not all(result.values()): raise ValueError('Provide train, validation AND test data')
    return result

if DATA_SOURCE=='csv':
    # Optional if stored on Drive:
    # from google.colab import drive
    # drive.mount('/content/drive')
    result=read_custom(CSV_PATH)
    train_rows,valid_rows,test_rows=(result[s] for s in ('train','validation','test'))

## 3 · Inspect and normalise carefully

**Good audio and consistent transcripts matter.** Listen and compare them. Long clips are omitted here solely to limit memory use. This does **not** perform VAD or diarisation, and it does not repair mismatched text/audio pairs.

We keep the source transcription (`text_original`) and create an ASR target (`text_target`). Our only optional text changes are **Unicode NFC** and trimming whitespace. We do not automatically delete Chao-tone symbols, diacritics, aspiration markers or punctuation.

In [ ]:
import unicodedata
from IPython.display import Audio as PlayAudio, display

def text_of(r):
    value=r['text_target']
    return unicodedata.normalize('NFC',value).strip() if NORMALISE_NFC else value

def eligible(rows):
    return [r for r in rows if 0.4<=len(r['audio'])/r['sr']<=MAX_SECONDS and text_of(r).strip()]
train_rows,valid_rows,test_rows=(eligible(x) for x in (train_rows,valid_rows,test_rows))
print('Eligible train, validation, test:',len(train_rows),len(valid_rows),len(test_rows))
assert train_rows and valid_rows and test_rows, 'Increase MAX_SECONDS or provide more aligned data'
print('Listen to one example:',text_of(test_rows[0]))
display(PlayAudio(test_rows[0]['audio'],rate=16000))

## 4 · Apply a pretrained MMS language adapter

This is **inference** with an existing model. The first run downloads a large checkpoint and may be slow. The code checks whether the selected MMS adapter language exists. If not, inspect the model card rather than assuming all language names correspond to an adapter.

For multilingual code-switching such as Mandarin/English class recordings, do **not** assume a single-language MMS adapter will transcribe both languages correctly.

In [ ]:
import gc, jiwer
from transformers import AutoProcessor, Wav2Vec2ForCTC, Wav2Vec2CTCTokenizer

if RUN_BASELINE:
    inventory=Wav2Vec2CTCTokenizer.from_pretrained(MMS_MODEL)
    available=set(inventory.vocab.keys())
    if MMS_LANG not in available:
        print('Adapter not found:',MMS_LANG, 'Examples:',sorted(available)[:25])
    else:
        base_proc=AutoProcessor.from_pretrained(MMS_MODEL,target_lang=MMS_LANG)
        base_model=Wav2Vec2ForCTC.from_pretrained(
            MMS_MODEL,target_lang=MMS_LANG,ignore_mismatched_sizes=True,
            low_cpu_mem_usage=True).to('cuda' if torch.cuda.is_available() else 'cpu').eval()
        audio=test_rows[0]['audio']
        inp=base_proc(audio,sampling_rate=16000,return_tensors='pt')
        inp={k:v.to(base_model.device) for k,v in inp.items()}
        with torch.inference_mode(): ids=base_model(**inp).logits.argmax(-1)
        hyp=base_proc.batch_decode(ids.cpu())[0]
        print('REFERENCE:',text_of(test_rows[0]))
        print('HYPOTHESIS:',hyp)
        print('CER for this one clip:',jiwer.cer(text_of(test_rows[0]),hyp))
        del base_model,base_proc
        gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()

## 5 · Construct a language-specific character vocabulary

To allow your own orthography (including tone letters and unusual diacritics), we create a new vocabulary **from training transcripts only**. We replace the pretrained output head and reinitialise adapters for training. This is *not* the same as merely loading an existing pretrained language adapter.

For a language with many characters (especially written Chinese), 24 short clips will leave large gaps in the vocabulary. Unseen validation/test characters become `[UNK]`. This exercise is not expected to produce high accuracy. For complex writing systems or combining marks, a character-per-codepoint tokeniser may not be ideal.

In [ ]:
import json
from transformers import Wav2Vec2FeatureExtractor, Wav2Vec2Processor
WORK=Path('/content/mms_student')
WORK.mkdir(exist_ok=True)
alphabet=sorted(set(''.join(text_of(r) for r in train_rows)) - {' '})
vocab={ch:i for i,ch in enumerate(alphabet)}
for token in ('|','[UNK]','[PAD]'):
    if token not in vocab: vocab[token]=len(vocab)
(WORK/'vocab.json').write_text(json.dumps(vocab,ensure_ascii=False,indent=2),encoding='utf8')
tokeniser=Wav2Vec2CTCTokenizer(str(WORK/'vocab.json'),unk_token='[UNK]',pad_token='[PAD]',word_delimiter_token='|')
extractor=Wav2Vec2FeatureExtractor(feature_size=1,sampling_rate=16000,padding_value=0.0,do_normalize=True,return_attention_mask=True)
processor=Wav2Vec2Processor(feature_extractor=extractor,tokenizer=tokeniser)
print('Vocabulary tokens:',len(tokeniser))
for name,rows in (('validation',valid_rows),('test',test_rows)):
    missing=set(''.join(text_of(r) for r in rows)) - set(alphabet) - {' '}
    if missing: print('WARNING: unseen',name,'characters:',sorted(missing)[:30])

## 6 · Turn recordings and transcriptions into CTC training examples

Each audio segment becomes input values; the matching human transcription becomes target token IDs. **CTC** learns alignment between the longer audio sequence and the shorter text sequence.

In [ ]:
def prepare(r):
    x=processor.feature_extractor(r['audio'],sampling_rate=16000).input_values[0]
    y=processor.tokenizer(text_of(r)).input_ids
    return {'input_values':x,'labels':y,'reference':text_of(r)}
train_ready,valid_ready,test_ready=([prepare(r) for r in rows] for rows in (train_rows,valid_rows,test_rows))

def ctc_collate(items):
    a=processor.feature_extractor.pad([{'input_values':x['input_values']} for x in items],padding=True,return_tensors='pt')
    b=processor.tokenizer.pad([{'input_ids':x['labels']} for x in items],padding=True,return_tensors='pt')
    a['labels']=b['input_ids'].masked_fill(b['attention_mask']!=1,-100)
    return a
print('Input shape:',ctc_collate(train_ready[:1])['input_values'].shape)

## 7 · Experimental MMS adapter fine-tuning (GPU)

Set `RUN_FINETUNING=True` in section 1 before running this cell. This follows Hugging Face's [adapter recipe](https://huggingface.co/blog/mms_adapters): reinitialise the small adapters and output head, freeze the 1B backbone and train only the adapters plus CTC classifier. **Batch size 1** and short clips help, but Colab may still run out of memory.

We use **FP16** mixed precision on CUDA. BF16 is a different 16-bit format with an FP32-like exponent range but lower fraction precision than FP16. Neither choice changes the transcription; numerical precision and transcription normalisation are separate questions.

In [ ]:
import itertools
from torch.utils.data import DataLoader
trained=None
if RUN_FINETUNING:
    if not torch.cuda.is_available(): raise RuntimeError('No GPU is available; the 1B training demo requires CUDA')
    trained=Wav2Vec2ForCTC.from_pretrained(
        MMS_MODEL,vocab_size=len(tokeniser),pad_token_id=tokeniser.pad_token_id,
        ignore_mismatched_sizes=True,ctc_loss_reduction='mean',ctc_zero_infinity=True,
        attention_dropout=0.0,hidden_dropout=0.0,feat_proj_dropout=0.0,
        layerdrop=0.0,low_cpu_mem_usage=True)
    trained.init_adapter_layers()
    trained.freeze_base_model()
    for parameter in trained._get_adapters().values(): parameter.requires_grad=True
    trained=trained.cuda().train()
    print('Trainable weights:',sum(p.numel() for p in trained.parameters() if p.requires_grad))
    loader=DataLoader(train_ready,batch_size=1,shuffle=True,collate_fn=ctc_collate)
    optimiser=torch.optim.AdamW([p for p in trained.parameters() if p.requires_grad],lr=LEARNING_RATE)
    scaler=torch.amp.GradScaler('cuda')
    batches=itertools.cycle(loader)
    for step in range(1,STEPS+1):
        batch={k:v.cuda() for k,v in next(batches).items()}
        optimiser.zero_grad(set_to_none=True)
        with torch.autocast('cuda',dtype=torch.float16): loss=trained(**batch).loss
        if not torch.isfinite(loss): raise RuntimeError('Non-finite loss: check input/target lengths and precision')
        scaler.scale(loss).backward()
        scaler.step(optimiser)
        scaler.update()
        print(f'Step {step}/{STEPS}: training loss {loss.item():.3f}')
    trained.eval()
else:
    print('Fine-tuning is disabled. The preparation and inference activities still work.')

## 8 · Evaluate CER on unseen audio

Compare **hypothesis** (machine output) with **reference** (human transcription). CER measures character-level substitutions, deletions and insertions divided by reference characters. A low aggregate CER may conceal systematic tone or aspiration errors. Do not compare CER across different normalisation standards without noting the difference.

Only evaluate the new head if it was **actually trained**. A randomly reinitialised classifier does not have meaningful transcription accuracy.

In [ ]:
if trained is not None:
    references,hypotheses=[],[]
    for row in test_ready:
        b=ctc_collate([row])
        inputs={k:v.cuda() for k,v in b.items() if k!='labels'}
        with torch.inference_mode(),torch.autocast('cuda',dtype=torch.float16):
            ids=trained(**inputs).logits.argmax(-1)
        hyp=processor.batch_decode(ids.cpu())[0]
        references.append(row['reference']); hypotheses.append(hyp)
        print('REFERENCE:',row['reference'], '| HYPOTHESIS:', hyp)
    print('Corpus CER:',jiwer.cer(references,hypotheses))
    with (WORK/'predictions.csv').open('w',newline='',encoding='utf8') as f:
        w=csv.writer(f); w.writerow(['reference','hypothesis']); w.writerows(zip(references,hypotheses))
    print('Saved',WORK/'predictions.csv')
else:
    print('No trained model in memory; review the existing MMS model output instead.')

## 9 · Save your adapted layer and target vocabulary

The large pretrained backbone remains available from Hugging Face. The following cell saves only **the trainable adapter/classifier parameters**, tokeniser and metadata. This archive is **not** a standalone ready-to-run full Hugging Face checkpoint. Restoring it requires the same base model, output vocabulary size and adapter configuration. Do not publish trained weights without checking source licences and consent.

In [ ]:
from safetensors.torch import save_file
import zipfile
if trained is not None:
    out=WORK/'adapter_export';out.mkdir(exist_ok=True)
    processor.save_pretrained(out)
    save_file({name:par.detach().cpu().float().contiguous() for name,par in trained._get_adapters().items()},out/'adapter_and_head.safetensors')
    (out/'info.json').write_text(json.dumps({'base':MMS_MODEL,'lang_label':MMS_LANG,'nfc':NORMALISE_NFC,'steps':STEPS,'note':'Manual adapter and CTC head; not standalone'},indent=2))
    zpath=WORK/'adapter_export.zip'
    with zipfile.ZipFile(zpath,'w',zipfile.ZIP_DEFLATED) as z:
        for f in out.iterdir():z.write(f,arcname=f.name)
    print('Saved',zpath)
    print('Download with: from google.colab import files; files.download(str(zpath))')
else:print('No fine-tuning was run, so there are no adapter weights to save.')

## 10 · Task, troubleshooting and reflection

**Discuss after practical:**

1. Where was your audio recorded, and how did you determine the **correct transcription tier**?
2. What did you keep and change when normalising? Why not simply remove tone, aspiration and punctuation?
3. Does the selected MMS adapter exist for your language? If not, what are the implications of training a fresh output vocabulary?
4. Are errors random or systematic? Could clearer audio, segmentation, consistent targets, speaker coverage, better decoding or more data help?
5. Does CER capture the errors most relevant to your linguistic question?

**Troubleshoot in this order:** error traceback and `--help` → official package README → Hugging Face model/dataset cards → GitHub Discussions / Issues → Stack Exchange → a trusted AI assistant. AI suggestions need checking. Never paste credentials or restricted recordings into public tools.

Common errors: `CUDA out of memory` (shorter clips; suitable GPU), absent `audio_path` (check files/Drive mount), unsupported MMS language code (consult card), unseen `[UNK]` characters (expand representative training vocabulary/data), unexpectedly high CER (check segmentation and ground truth), or Colab disconnection (GPU limits vary).

**Post-session reflection (200–500 words):** state the chosen language, FLEURS dataset configuration or corpus/recording provenance, MMS adapter code, training/evaluation splits, code/settings, whether you attempted fine-tuning, what errors occurred, what helped, and how you would improve your model. A low-performing model or lack of free GPU access can still be analysed rigorously.

References: [FLEURS](https://huggingface.co/datasets/google/fleurs) · [MMS documentation](https://huggingface.co/docs/transformers/model_doc/mms) · [MMS fine-tuning tutorial](https://huggingface.co/blog/mms_adapters) · [ASR starter repository](https://github.com/juanpalm-pixel/asr-work).